In [1]:
import pandas as pd
import numpy as np
import os
import glob
from cleaning_script import adaptive_clean_real_estate_data
from feature_engineering import build_features

In [26]:
#GOING BACK AND LOOKING AT THE RAW DATASETS
data_dir = '../original_datasets'

# Pattern to match all CRMLS monthly files
pattern = os.path.join(data_dir, "CRMLSSold2025??_filled.csv")

# Get all matching file paths
file_list = sorted(glob.glob(pattern))

print("Files found:", file_list)

# Load and store each DataFrame
dfs = []
for f in file_list:
    df = pd.read_csv(f, low_memory=False)
    dfs.append(df)

# Merge vertically (stack rows)
merged_df = pd.concat(dfs, ignore_index=True)

print("Merged shape:", merged_df.shape)



Files found: ['../original_datasets\\CRMLSSold202502_filled.csv', '../original_datasets\\CRMLSSold202503_filled.csv', '../original_datasets\\CRMLSSold202504_filled.csv', '../original_datasets\\CRMLSSold202505_filled.csv', '../original_datasets\\CRMLSSold202506_filled.csv', '../original_datasets\\CRMLSSold202507_filled.csv']
Merged shape: (133092, 80)


In [27]:
aug = pd.read_csv('../original_datasets/CRMLSSold202508_filled-2.csv')
sept = pd.read_csv('../original_datasets/CRMLSSold202509.csv')
merged_df

,BuyerAgentAOR,ListAgentAOR,Flooring,ViewYN,WaterfrontYN,BasementYN,PoolPrivateYN,OriginalListPrice,ListingKey,ListAgentEmail,...,MainLevelBedrooms,NewConstructionYN,GarageSpaces,HighSchoolDistrict,PostalCode,AssociationFee,LotSizeSquareFeet,MiddleOrJuniorSchoolDistrict,latfilled,lonfilled
0,RanchoSoutheast,RanchoSoutheast,NaN,True,NaN,NaN,NaN,60000.0,526199946,cmark1018@yahoo.com,...,NaN,False,NaN,NaN,92307,0.0,28000.0,NaN,False,False
1,InlandValleys,InlandValleys,NaN,False,NaN,NaN,NaN,550000.0,525585060,mozcorona@aol.com,...,NaN,False,NaN,NaN,92553,0.0,39640.0,NaN,False,False
2,SanDiego,SanDiego,NaN,False,NaN,NaN,False,880000.0,497696903,lenskab@gmail.com,...,NaN,False,2.0,NaN,91942,NaN,NaN,NaN,False,False
3,SanDiego,SanDiego,NaN,False,NaN,NaN,False,875000.0,497696407,lenskab@gmail.com,...,NaN,False,2.0,NaN,91942,NaN,NaN,NaN,False,False
4,SanDiego,SanDiego,NaN,False,NaN,NaN,False,849000.0,486616176,lenskab@gmail.com,...,NaN,False,2.0,NaN,91942,NaN,NaN,NaN,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
133087,OrangeCounty,OrangeCounty,NaN,True,NaN,NaN,NaN,7750000.0,1022700795,adam@OCcollective.com,...,NaN,False,NaN,NaN,92154,0.0,243065.0,NaN,False,False
133088,CaliforniaDesert,CaliforniaDesert,"Carpet,Tile",True,NaN,NaN,True,5800.0,1020000387,avi@unigrafix.com,...,NaN,False,2.0,NaN,92270,775.0,5227.0,NaN,False,False
133089,Glendale,Glendale,NaN,NaN,NaN,NaN,NaN,13850000.0,1019043037,Listings@LockerRealty.com,...,NaN,NaN,NaN,NaN,90277,NaN,68003.0,NaN,False,False
133090,OrangeCounty,OrangeCounty,NaN,True,NaN,NaN,NaN,2500000.0,1018313206,drew@teicheirateam.com,...,NaN,False,NaN,NaN,91730,0.0,1310720.0,NaN,False,False


In [28]:
merged_df = pd.concat([merged_df, aug, sept], ignore_index=True)
merged_df.shape

(178507, 80)

In [29]:
merged_df = merged_df[(merged_df['PropertyType']=='Residential') & (merged_df['PropertySubType']=='SingleFamilyResidence')]


In [30]:
merged_df.to_csv('combined_raw_data.csv', index=False)

In [31]:
testing_set = pd.read_csv('../original_datasets/CRMLSSold202510.csv')
testing_set.to_csv('testing_raw_data.csv', index=False)

In [ ]:
train_cleaned, stats, mechs = adaptive_clean_real_estate_data(merged_df, fit = True)
test_clean, _, _ = adaptive_clean_real_estate_data(testing_set, fit = False, imputation_stats=stats, missing_mechanisms=mechs, trim_outliers=True)

ADAPTIVE CLEANING - TRAINING DATA
Initial shape: (89843, 81)

Dropping 21 columns with >75% missing

DIAGNOSING MISSING DATA MECHANISMS

ViewYN: NMAR (amenity - missing likely means absent)

PoolPrivateYN: NMAR (amenity - missing likely means absent)

AttachedGarageYN: NMAR (amenity - missing likely means absent)

LotSizeAcres: MAR detected (correlates with 1 features)
  - Latitude: r=-0.135

BathroomsTotalInteger: MAR detected (correlates with 1 features)
  - LotSizeSquareFeet: r=0.104

FireplaceYN: NMAR (amenity - missing likely means absent)

Stories: MAR detected (correlates with 3 features)
  - Latitude: r=0.299
  - Longitude: r=-0.220
  - BathroomsTotalInteger: r=0.100

LotSizeArea: MAR detected (correlates with 1 features)
  - Latitude: r=-0.136

GarageSpaces: MAR detected (correlates with 1 features)
  - YearBuilt: r=-0.147

LotSizeSquareFeet: MAR detected (correlates with 2 features)
  - Latitude: r=-0.135
  - LotSizeAcres: r=0.871

--------------------------------------------

In [23]:
train_cleaned.to_csv('train_cleaned.csv', index=False)
test_clean.to_csv('test_cleaned.csv', index=False)

GOAL: BRAINSTORM NEIGHBORHOOD LEVEL FEATURES

In [19]:
train_with_features, test_with_features = build_features(train_cleaned, test_clean)

Building features...
  - Temporal features
  - Log transforms and age buckets
  - Ratio features
  - Boolean interactions
  - Coastal distance
  - Geographic clustering
  - Target encoding (k-fold)
  - Local pricing benchmarks
  - Market velocity indicators
  - Cleanup
✓ Feature engineering complete
  Train: (66871, 60)
  Test:  (4916, 60)
  ⚠️  Warning: Categorical columns in train: ['SourceFile']
  ⚠️  Warning: Categorical columns in test: ['CoListOfficeName']


In [20]:
train_with_features

,ViewYN,PoolPrivateYN,ClosePrice,Latitude,Longitude,LivingArea,AttachedGarageYN,ParkingTotal,LotSizeAcres,YearBuilt,...,IsCoastal,GeoCluster,City_TE,PostalCode_TE,CountyOrParish_TE,ZIP_MedianPPSF,City_MedianPPSF,County_MedianPPSF,ZIP_SalesCount,ZIP_MedianDOM
2,False,False,875000.0,32.765380,-117.043486,2340.0,True,6.0,10.335832,2021.0,...,0,7,4.655753e+06,9.117890e+05,1.658581e+06,639.820548,603.876289,605.345912,12,82.5
3,False,False,875000.0,32.765038,-117.043568,2165.0,True,4.0,10.250255,2021.0,...,0,7,4.882246e+06,9.023506e+05,1.679895e+06,639.820548,603.876289,605.345912,12,82.5
4,False,False,849000.0,32.765031,-117.043252,2158.0,True,4.0,10.248912,2021.0,...,0,7,1.014663e+06,9.060148e+05,1.568255e+06,639.820548,603.876289,605.345912,12,82.5
15,True,False,1100000.0,33.724000,-118.294924,2545.0,False,3.0,0.103300,1962.0,...,0,0,1.087204e+06,9.945915e+05,1.540484e+06,669.701987,658.866995,645.624103,15,51.0
16,False,False,760000.0,37.687556,-122.150114,1692.0,True,2.0,0.115700,1950.0,...,0,1,9.222381e+05,9.349555e+05,1.463035e+06,659.484938,638.103920,765.172861,9,31.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
133047,True,True,2625000.0,33.602236,-116.147017,7513.0,True,10.0,0.320000,2019.0,...,0,11,4.280677e+05,4.280677e+05,7.793481e+05,227.715877,227.715877,323.834197,6,177.0
133062,True,False,375000.0,34.118035,-116.428352,1770.0,True,2.0,0.095200,2022.0,...,0,11,4.300089e+05,4.298940e+05,6.103156e+05,254.922942,254.355401,328.194005,49,91.0
133068,True,False,1980000.0,39.357862,-123.821861,4700.0,False,6.0,0.560000,1979.0,...,0,18,1.531056e+06,1.531056e+06,8.604704e+05,514.904687,514.904687,454.480146,4,343.5
133073,True,False,550000.0,34.481528,-117.266840,3045.0,True,2.0,0.165300,1990.0,...,0,17,4.440091e+05,4.367601e+05,6.082315e+05,257.308762,244.097565,328.194005,33,80.0


import model_training

In [2]:
import model_training

c:\Users\jueeh\idx-exchange-ds34\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
model, metrics = model_training.main()


XGBOOST REAL ESTATE PRICE PREDICTION PIPELINE
Loading data...
ADAPTIVE CLEANING - TRAINING DATA
Initial shape: (89843, 80)

Dropping 21 columns with >75% missing

DIAGNOSING MISSING DATA MECHANISMS

ViewYN: NMAR (amenity - missing likely means absent)

PoolPrivateYN: NMAR (amenity - missing likely means absent)

AttachedGarageYN: NMAR (amenity - missing likely means absent)

LotSizeAcres: MAR detected (correlates with 1 features)
  - Latitude: r=-0.135

BathroomsTotalInteger: MAR detected (correlates with 1 features)
  - LotSizeSquareFeet: r=0.104

FireplaceYN: NMAR (amenity - missing likely means absent)

Stories: MAR detected (correlates with 3 features)
  - Latitude: r=0.299
  - Longitude: r=-0.220
  - BathroomsTotalInteger: r=0.100

LotSizeArea: MAR detected (correlates with 1 features)
  - Latitude: r=-0.136

GarageSpaces: MAR detected (correlates with 1 features)
  - YearBuilt: r=-0.147

LotSizeSquareFeet: MAR detected (correlates with 2 features)
  - Latitude: r=-0.135
  - LotSi

[I 2025-11-25 14:19:21,533] A new study created in memory with name: no-name-a9acc7b1-bbdc-469e-8703-e254970050b8


  - Cleanup
✓ Feature engineering complete
  Train: (89775, 59)
  Test:  (11905, 59)
  Train shape after feature engineering: (89775, 59)
  Test shape after feature engineering:  (11905, 59)
  ✓ All features are numeric

Final shapes:
  X_train: (89775, 58)
  y_train: (89775,)
  X_test:  (11905, 58)
  ✓ Feature alignment verified: 58 features

HYPERPARAMETER TUNING (40 trials, 3-fold CV)


Best trial: 0. Best value: -0.0719286:   2%|▎         | 1/40 [00:10<06:38, 10.21s/it]

[I 2025-11-25 14:19:31,742] Trial 0 finished with value: -0.0719285938286703 and parameters: {'n_estimators': 1005, 'learning_rate': 0.015996747560492682, 'max_depth': 6, 'min_child_weight': 5, 'subsample': 0.6464485640585839, 'colsample_bytree': 0.8966709871805307, 'gamma': 2.586679419679114, 'reg_alpha': 1.116105598008624, 'reg_lambda': 2.8756851154283174}. Best is trial 0 with value: -0.0719285938286703.


...

[I 2025-11-25 14:24:03,564] Trial 37 pruned. 


Best trial: 7. Best value: 0.0111063:  98%|█████████▊| 39/40 [04:45<00:03,  3.53s/it]

[I 2025-11-25 14:24:07,047] Trial 38 pruned. 


Best trial: 7. Best value: 0.0111063: 100%|██████████| 40/40 [04:48<00:00,  7.21s/it]


[I 2025-11-25 14:24:09,858] Trial 39 pruned. 

✓ Tuning complete!
Best R²: 0.0111

Best Parameters:
  n_estimators: 415
  learning_rate: 0.010623610217812942
  max_depth: 5
  min_child_weight: 8
  subsample: 0.6814827938269733
  colsample_bytree: 0.6545804133018197
  gamma: 3.332060749158567
  reg_alpha: 2.2050235944871517
  reg_lambda: 1.0697925535421349

TRAINING FINAL MODEL
Train size: 71820
Validation size: 17955

Training model...
FINAL MODEL METRICS

Training Set:
  R²:     0.2552
  MAPE:   8290039265206894592.00%
  MdAPE:  15.00%

Validation Set:
  R²:     0.0062
  MAPE:   34.48%
  MdAPE:  14.90%

Overfitting Check:
  R² difference: 0.2490
  ⚠️  Significant overfitting detected
TEST SET EVALUATION

Test Set Metrics:
  R²:     -1.3866
  MAPE:   37.89%
  MdAPE:  15.88%

SAVING ARTIFACTS
✓ Model saved: xgb_final_model.pkl
✓ Parameters saved: best_params.pkl
✓ Predictions saved: test_predictions.csv
✓ Metrics saved: model_metrics.csv

PIPELINE COMPLETE!
